# Train Remaining Health Risk Models

This notebook mirrors the shared project pipeline and documents how the remaining targets are trained using the same feature space as the existing models.

In [ ]:
import os
import sys
import pandas as pd

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocess import load_dataset, select_features, encode_categoricals, separate_features_targets, TARGET_COLUMNS
from src.features import fit_scaler, transform_features
from src.train import train_candidate_models
from src.evaluate import calculate_metrics, compare_models


In [ ]:
df = load_dataset(os.path.join(PROJECT_ROOT, 'data', 'raw_data.csv'))
df = encode_categoricals(select_features(df))
X, targets = separate_features_targets(df)
target_df = pd.DataFrame(targets)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train_df, y_test_df = train_test_split(X, target_df, test_size=0.2, random_state=42)
scaler = fit_scaler(X_train)
X_train_scaled = transform_features(scaler, X_train)
X_test_scaled = transform_features(scaler, X_test)

summary = []
for target in ['hypertension_risk_score', 'obesity_risk_score', 'cholesterol_risk_score']:
    models = train_candidate_models(X_train_scaled, y_train_df[target])
    metrics = {name: calculate_metrics(y_test_df[target], model.predict(X_test_scaled)) for name, model in models.items()}
    best = compare_models(metrics['LinearRegression'], metrics['DecisionTree'], 'LinearRegression', 'DecisionTree')
    summary.append({'target': target, 'best_model': best, **{f'{name}_R2': vals['R2'] for name, vals in metrics.items()}})

pd.DataFrame(summary)
